In [1]:
import numpy as np
import pandas as pd
import json
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, Binarizer
from sklearn import datasets, metrics, svm
from sklearn.linear_model import SGDClassifier, Lasso
from sklearn.svm import LinearSVC, NuSVC
from sklearn.cluster import KMeans, MeanShift
from sklearn.naive_bayes import BernoulliNB, GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
# from sklearn.tree import DecisionTreeClassifier
# from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, GradientBoostingClassifier
# from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
import torch
import ezkl
import os
from torch import nn
from hummingbird.ml import convert, constants
import hummingbird.ml

In [2]:
odroid1 = pd.read_csv("https://raw.githubusercontent.com/brite3001/drawnapart/master/onscreen/odroid1_firefox_onscreen.csv")
odroid2 = pd.read_csv("https://raw.githubusercontent.com/brite3001/drawnapart/master/onscreen/odroid2_firefox_onscreen.csv")
odroid3 = pd.read_csv("https://raw.githubusercontent.com/brite3001/drawnapart/master/onscreen/odroid3_firefox_onscreen.csv")
odroid4 = pd.read_csv("https://raw.githubusercontent.com/brite3001/drawnapart/master/onscreen/odroid4_firefox_onscreen.csv")

In [3]:
rpi4a = pd.read_csv("https://raw.githubusercontent.com/brite3001/drawnapart/master/onscreen/rpi-4a_chrome_onscreen.csv")
rpi8a = pd.read_csv("https://raw.githubusercontent.com/brite3001/drawnapart/master/onscreen/rpi-8a_chrome_onscreen.csv")
rpi8b = pd.read_csv("https://raw.githubusercontent.com/brite3001/drawnapart/master/onscreen/rpi-8b_chrome_onscreen.csv")
rpi8c = pd.read_csv("https://raw.githubusercontent.com/brite3001/drawnapart/master/onscreen/rpi-8c_chrome_onscreen.csv")
rpi8d = pd.read_csv("https://raw.githubusercontent.com/brite3001/drawnapart/master/onscreen/rpi-8d_chrome_onscreen.csv")

In [4]:
df = pd.concat([odroid1[:1000], odroid2[:1000], odroid3[:1000], odroid4[:1000], rpi4a[:1000], rpi8a[:1000], rpi8b[:1000], rpi8c[:1000], rpi8d[:1000]])
df = df.sample(frac=1).reset_index(drop=True)

In [5]:
df.head()

,label,Feature 0,Feature 1,Feature 2,Feature 3,Feature 4,Feature 5,Feature 6
0,rpi-8a,228.5,232.9,433.3,16.7,232.9,233.2,433.4
1,rpi-8c,69.7,961.1,1692.8,17.4,18.9,41.6,94.7
2,rpi-8d,17.8,20.1,18.4,969.6,18.7,21.0,21.0
3,rpi-8a,231.5,430.3,15.7,232.7,233.3,233.1,433.6
4,rpi-8d,418.3,415.4,15.8,17.1,17.4,17.1,16.5


# Data Prep and model training

In [6]:
encoder = LabelEncoder()
encoded_labels = pd.DataFrame(encoder.fit_transform(df['label']))

In [7]:
label_map = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))

In [8]:
label_map 


{'odroid1': 0,
 'odroid2': 1,
 'odroid3': 2,
 'odroid4': 3,
 'rpi-4a': 4,
 'rpi-8a': 5,
 'rpi-8b': 6,
 'rpi-8c': 7,
 'rpi-8d': 8}

In [9]:
df = df.drop(['label'], axis=1)

In [10]:
df.head()

,Feature 0,Feature 1,Feature 2,Feature 3,Feature 4,Feature 5,Feature 6
0,228.5,232.9,433.3,16.7,232.9,233.2,433.4
1,69.7,961.1,1692.8,17.4,18.9,41.6,94.7
2,17.8,20.1,18.4,969.6,18.7,21.0,21.0
3,231.5,430.3,15.7,232.7,233.3,233.1,433.6
4,418.3,415.4,15.8,17.1,17.4,17.1,16.5


In [11]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, MaxAbsScaler, QuantileTransformer, PowerTransformer

scaler = StandardScaler()
scaled_features = pd.DataFrame(scaler.fit_transform(df))

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    scaled_features, encoded_labels, test_size=0.25, shuffle=False
)

In [13]:
scaled_features

,0,1,2,3,4,5,6
0,0.245409,0.104508,0.416932,-0.749307,0.290049,0.447182,1.493036
1,-0.405980,2.629137,3.895028,-0.746883,-0.705498,-0.603474,-0.340375
2,-0.618871,-0.633258,-0.728809,2.550314,-0.706428,-0.716436,-0.739319
3,0.257715,0.788883,-0.736266,-0.001360,0.291910,0.446634,1.494118
4,1.023959,0.737225,-0.735989,-0.747922,-0.712476,-0.737822,-0.763677
...,...,...,...,...,...,...,...
8995,-0.019166,-0.123964,-0.318452,-0.228861,-0.021176,0.078685,0.056404
8996,-0.635689,0.042796,-0.183692,-0.115630,1.143707,0.447730,0.408796
8997,-0.617641,-0.635338,3.096405,-0.734417,-0.695263,-0.732887,-0.747980
8998,-0.019166,-0.092761,-0.354352,-0.215010,0.053258,0.084168,0.050991


In [14]:
# clf = RandomForestClassifier()
# clf = Gbc()
# clf = KNeighborsClassifier()
# clf = LGBMClassifier()
# clf = GaussianNB()
clf = XGBClassifier()
# clf = SGDClassifier(max_iter=1000, tol=1e-3)
# clf = LinearSVC(random_state=1, tol=1e-05)
# clf = NuSVC() # 0.74 acc # converting takes a while
# clf = KMeans(n_clusters=9, random_state=0, n_init="auto")
# clf = MeanShift(bandwidth=9)
# clf = BernoulliNB()
# clf = GaussianNB()
# clf = KNeighborsClassifier(n_neighbors=3)
# clf = MLPClassifier(random_state=1, max_iter=1000) # needs 18.7gb ram, 0.78 accuracy

In [15]:
clf.fit(X_train.values, y_train.values)

# Predict the value of the digit on the test subset
predicted = clf.predict(X_test.values)

In [16]:
print(metrics.classification_report(y_test, predicted))

              precision    recall  f1-score   support

           0       0.95      0.95      0.95       278
           1       0.99      0.99      0.99       239
           2       0.94      0.92      0.93       263
           3       0.99      0.98      0.99       235
           4       0.88      0.91      0.90       259
           5       0.82      0.88      0.85       238
           6       0.85      0.82      0.83       257
           7       0.70      0.68      0.69       239
           8       0.74      0.71      0.73       242

    accuracy                           0.87      2250
   macro avg       0.87      0.87      0.87      2250
weighted avg       0.87      0.87      0.87      2250



## Convert Sklearn -> pytorch

In [17]:
extra_config = {hummingbird.ml.operator_converters.constants.BATCH_SIZE: 1}

# Convert the DataFrame to a numpy array and keep column names
X_test_array = X_test.values

clf_as_torch = convert(clf, "torch", X_test_array[:1].copy(), extra_config=extra_config)
# assert predictions from torch are = to sklearn
diffs = []

for i in range(len(X_test_array)):
    torch_pred = clf_as_torch.predict(torch.tensor(X_test_array[i].reshape(1, -1)))
    sk_pred = clf.predict(X_test_array[i].reshape(1, -1))
    diffs.append(torch_pred[0] != sk_pred[0])


In [18]:
sum(diffs)

0

# EZKL Starts Here:

In [19]:
model_path = os.path.join('network.onnx')
compiled_model_path = os.path.join('network.compiled')
pk_path = os.path.join('test.pk')
vk_path = os.path.join('test.vk')
settings_path = os.path.join('settings.json')

witness_path = os.path.join('witness.json')
data_path = os.path.join('input.json')

In [20]:
# Input to the model
# Convert DataFrame to numpy array
x = X_train.values[0]

shape = x.shape
print(shape)
torch_out = clf_as_torch.predict(torch.tensor(x.reshape(1, -1)))
# Export the model
torch.onnx.export(clf_as_torch.model,               # model being run
                  torch.tensor(x.reshape(1, -1)), # model input (or a tuple for multiple inputs)
                  "network.onnx", # where to save the model (can be a file or file-like object)
                  export_params=True,        # store the trained parameter weights inside the model file
                  opset_version=18,          # the ONNX version to export the model to
                  input_names=['input'],   # the model's input names
                  output_names=['output'],  # the model's output names
                  dynamic_axes={'input': {0: 'batch_size'},    # variable length axes
                                'output': {0: 'batch_size'}})

d = x.reshape([-1]).tolist()

data = dict(input_shapes=[shape],
            input_data=[d],
            output_data=[o.reshape([-1]).tolist() for o in torch_out])

# Serialize data into file:
json.dump(data, open("input.json", 'w'))

(7,)


In [21]:
run_args = ezkl.PyRunArgs()
# run_args.variables = [("batch_size", 1)]
# run_args.input_visibility = "hashed"
# run_args.output_visibility = "public"
# run_args.input_scale = 7
# run_args.decomp_legs = 4
# run_args.logrows = 14

# TODO: Dictionary outputs
res = ezkl.gen_settings(model_path, settings_path, py_run_args=run_args)
assert res == True


In [22]:
# 30s runtime
# 27s with zscore data
cal_path = os.path.join("calibration.json")

data_array = (torch.rand(20,*shape).detach().numpy()).reshape([-1]).tolist()

data = dict(input_data = [data_array])

# Serialize data into file:
json.dump(data, open(cal_path, 'w'))

# "resources/col-overflow"
res = await ezkl.calibrate_settings(data_path, model_path, settings_path)
assert res == True



 <------------- Numerical Fidelity Report (input_scale: 12, param_scale: 12, scale_input_multiplier: 10) ------------->

+--------------------+-----------------+----------------+-----------------+-----------------+------------------+----------------+---------------+---------------------+--------------------+------------------------+
| mean_error         | median_error    | max_error      | min_error       | mean_abs_error  | median_abs_error | max_abs_error  | min_abs_error | mean_squared_error  | mean_percent_error | mean_abs_percent_error |
+--------------------+-----------------+----------------+-----------------+-----------------+------------------+----------------+---------------+---------------------+--------------------+------------------------+
| -0.000000016817557 | 0.0000009060695 | 0.000017906239 | -0.000028252602 | 0.0000056337026 | 0.0000009060695  | 0.000028252602 | 0             | 0.00000000011646159 | 0.79999715         | 0.8000029              |
+--------------------

In [24]:
# srs path
res = await ezkl.get_srs(settings_path)

In [25]:
res = ezkl.compile_circuit(model_path, compiled_model_path, settings_path)
assert res == True

In [26]:
# 30s 6GB ram
# run_args.input_scale = 7, run_args.logrows = 10

# 4GB with resources/col-overflow in calibrate
res = ezkl.setup(
        compiled_model_path,
        vk_path,
        pk_path,  
    )

assert res == True
assert os.path.isfile(vk_path) # verification key
assert os.path.isfile(pk_path) # proving key
assert os.path.isfile(settings_path) # circuit settings

In [27]:
# now generate the witness file 


res = await ezkl.gen_witness(data_path, compiled_model_path, witness_path)
assert os.path.isfile(witness_path)

In [72]:
proof_path = os.path.join('test.pf')


# 18s
# NEEDED ~7GB FOR XGBOOST run_args.input_scale = 7

res = ezkl.prove(
        witness_path,
        compiled_model_path,
        pk_path,
        proof_path,
        
        "single",
    )

assert os.path.isfile(proof_path)

In [28]:
# VERIFY IT
# NEEDED ALMOST NO RESOURCES FOR XGBOOST

# The inputs to (non-EVM) verify are:

# the proof file
# the verification key
# the circuit settings, and
# the structured reference string


res = ezkl.verify(
        proof_path,
        settings_path,
        vk_path,
        
    )

assert res == True
print("verified")

verified
